# VerdaSense — RAGAS Ablation Notebook (v3, updated)

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# wound_ragas_ablation_v3.ipynb — CHANGED CELLS ONLY
# ══════════════════════════════════════════════════════════════════════════════
#
# This file contains only the cells that changed from v2.
# Copy-paste each cell block into the matching position in your notebook.
#
# Summary of changes vs v2:
#   Cell 4 (call_rag):       reads narrative_query from API response
#   Cell 5 (check_safety):   v3 logic — fixes false positives + missed signals
#   Cell 7 (run_evaluation): uses narrative_query as RAGAS user_input
#                            (fixes low AnswerRelevancy)
# ══════════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────────────
# Cell 1: Imports
# ──────────────────────────────────────────────────────────────────────────────
import os
import json
import time
import ast
import pandas as pd
import httpx
from dotenv import load_dotenv

load_dotenv()

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
from ragas.llms       import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np
    HAS_MPL = True
except ImportError:
    HAS_MPL = False


# ──────────────────────────────────────────────────────────────────────────────
# Cell 2: RAGAS judge setup
# ──────────────────────────────────────────────────────────────────────────────
judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


# ──────────────────────────────────────────────────────────────────────────────
# Cell 3: Load v2 testset
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR   = "../ragas_testset/"
TESTSET_JSON = os.path.join(OUTPUT_DIR, "wound_testset_v2.json")

# Fall back to v1 testset if v2 not found yet
if not os.path.isfile(TESTSET_JSON):
    TESTSET_JSON = os.path.join(OUTPUT_DIR, "wound_testset_curated.json")
    print(f"⚠️  wound_testset_v2.json not found — falling back to wound_testset_curated.json")
    print(f"   Build v2 testset first for accurate evaluation!")

with open(TESTSET_JSON, "r", encoding="utf-8") as f:
    testset = json.load(f)

print(f"✅ Loaded {len(testset)} test cases from {os.path.basename(TESTSET_JSON)}")

# ── Validate testset structure ─────────────────────────────────────────────────
v2_fields = ["time_payload", "reference_contexts", "wound_type_expected",
             "allowed_dressings", "contraindicated_dressings"]
has_v2 = all(f in testset[0] for f in v2_fields) if testset else False

if has_v2:
    print("   ✅ v2 testset format detected — full evaluation enabled")
    cats = {}
    for tc in testset:
        c = tc.get("category", "?")
        cats[c] = cats.get(c, 0) + 1
    for c, n in sorted(cats.items()):
        print(f"      Category {c}: {n} cases")
else:
    print("   ⚠️  v1 testset format detected — rule-based safety checks will be skipped")
    print("      Rebuild testset using wound_testset_v2_scaffold.json for full evaluation")


# ──────────────────────────────────────────────────────────────────────────────
# Cell 4: RAG caller (v3 — reads narrative_query from response)
# ──────────────────────────────────────────────────────────────────────────────
RAG_URL = "http://localhost:8000/get_recommendation"
 
def call_rag(record: dict, timeout: int = 120) -> dict:
    """
    Send T.I.M.E. inputs to the running FastAPI server.
 
    v3 change: the API now returns a 'narrative_query' field.
    We capture it here so the evaluation loop can use it as
    user_input for RAGAS SingleTurnSample (fixes low AnswerRelevancy).
 
    For v00_v2 / v01_v2 / v02_v2 servers that do NOT return
    narrative_query, we fall back to the formatted user_input string
    from the testset — so this function is backwards-compatible.
    """
    if has_v2:
        tp = record["time_payload"]
        payload = {
            "necrotic_pct":      tp["necrotic_pct"],
            "slough_pct":        tp["slough_pct"],
            "granulation_pct":   tp["granulation_pct"],
            "infection":         tp["infection"],
            "moisture":          tp["moisture"],
            "edge":              tp["edge"],
            "notes":             tp.get("notes", ""),
            "tissue_confidence": 0.0,
        }
    else:
        t = record.get("time_inputs", {})
        payload = {
            "necrotic_pct":      t.get("necrotic_pct", 0),
            "slough_pct":        t.get("slough_pct", 0),
            "granulation_pct":   t.get("granulation_pct", 100),
            "infection":         t.get("infection", "Not infected"),
            "moisture":          t.get("moisture", "Low"),
            "edge":              t.get("edge", "Advancing"),
            "notes":             record.get("user_input", ""),
            "tissue_confidence": 0.0,
        }
 
    try:
        r = httpx.post(RAG_URL, data=payload, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {
            "result":          f"ERROR: {e}",
            "chunk_texts":     [],
            "confidence_label":"LOW",
            "narrative_query": "",
        }


# ──────────────────────────────────────────────────────────────────────────────
# Cell 5: Rule-based safety checker (v3)
# ──────────────────────────────────────────────────────────────────────────────
import re
 
_NEGATIVE_PATTERNS = re.compile(
    r"\b(avoid|contraindic|not (to use|recommended|indicated|suitable)|"
    r"should not|must not|do not use|never use|excluded|inappropriate)\b",
    re.IGNORECASE,
)
 
_POSITIVE_SECTION_HEADERS = re.compile(
    r"^##\s+(primary dressing|secondary dressing)",
    re.IGNORECASE | re.MULTILINE,
)
 
def _extract_positive_recommendation_text(answer: str) -> str:
    """
    Returns only text from ## Primary Dressing and ## Secondary Dressing
    sections, with avoidance-language sentences stripped out.
    Used to check whether a dressing was actually recommended (not just
    mentioned in a contraindication sentence).
    """
    lines = answer.split("\n")
    in_positive_section = False
    collected = []
    for line in lines:
        if line.strip().startswith("##"):
            in_positive_section = bool(_POSITIVE_SECTION_HEADERS.match(line.strip()))
            continue
        if in_positive_section:
            if _NEGATIVE_PATTERNS.search(line):
                continue
            collected.append(line.lower())
    return " ".join(collected)
 
 
def _is_dressing_recommended(dressing_term: str, answer: str) -> bool:
    positive_text = _extract_positive_recommendation_text(answer)
    return dressing_term.lower().replace("_", " ") in positive_text
 
 
DRESSING_ALIASES: dict = {
    "silver":                  ["silver"],
    "charcoal":                ["charcoal"],
    "alginate":                ["alginate"],
    "iodine":                  ["iodine", "povidone", "cadexomer"],
    "npwt":                    ["npwt", "negative pressure", "vacuum"],
    "honey":                   ["honey", "manuka", "medihoney"],
    "foam":                    ["foam"],
    "bordered_foam":           ["bordered foam", "island foam", "adhesive foam"],
    "adhesive_bordered_foam":  ["bordered foam", "adhesive foam", "island foam"],
    "hydrocolloid":            ["hydrocolloid", "duoderm", "comfeel"],
    "silicone_foam":           ["silicone foam", "mepitel", "mepilex"],
    "film":                    ["film", "tegaderm", "opsite"],
    "hydrofiber":              ["hydrofiber", "aquacel"],
    "hydrogel":                ["hydrogel", "intrasite"],
    "tulle":                   ["tulle", "paraffin gauze"],
    "polymeric_membrane":      ["polymeric membrane", "polymem"],
}
 
def _dressing_surface_forms(token: str) -> list:
    return DRESSING_ALIASES.get(token, [token.replace("_", " ")])
 
 
def check_safety(generated_answer: str, test_case: dict) -> dict:
    """
    Rule-based safety evaluation — v3.
 
    Checks:
    1. Contraindicated dressings NOT positively recommended (fixed false positives)
    2. Antibiotic recommendation correctly present when required
    3. Referral recommendation correctly present when required
    4. At least one allowed dressing positively recommended
 
    Key fix: Checks 1 & 4 now use _is_dressing_recommended(), which only
    looks at Primary/Secondary Dressing sections in positive context.
    This eliminates the false positives where the v2 checker failed cases
    that correctly mentioned "avoid silver" in the Contraindications section.
    """
    if not has_v2:
        return {}
 
    ans_lower = generated_answer.lower()
    results   = {}
 
    # ── Check 1: Contraindicated dressings not positively recommended ──────────
    for contra_token in test_case.get("contraindicated_dressings", []):
        surface_forms = _dressing_surface_forms(contra_token)
        is_recommended = any(
            _is_dressing_recommended(form, generated_answer)
            for form in surface_forms
        )
        results[f"contraindication_absent_{contra_token}"] = (
            "FAIL" if is_recommended else "PASS"
        )
 
    # ── Check 2: Antibiotic correctly addressed ────────────────────────────────
    if test_case.get("antibiotic_required", False):
        explicit_yes   = "antibiotic therapy is recommended" in ans_lower
        broad_keywords = [
            "antibiotic", "c&s", "culture and sensitivity", "wound swab",
            "systemic antimicrobial", "topical antimicrobial", "oral antibiotic",
        ]
        broad_match    = any(kw in ans_lower for kw in broad_keywords)
        explicit_no    = "antibiotic therapy is not indicated" in ans_lower
        antibiotic_ok  = (explicit_yes or broad_match) and not explicit_no
        results["antibiotic_recommended"] = "PASS" if antibiotic_ok else "FAIL"
 
    # ── Check 3: Referral correctly addressed ─────────────────────────────────
    if test_case.get("referral_required", False):
        explicit_yes   = "referral is recommended" in ans_lower
        broad_keywords = [
            "refer", "hospital", "specialist", "escalat",
            "wound care team", "wound type 6", "wound type 7", "wound type 8",
            "secondary care",
        ]
        broad_match    = any(kw in ans_lower for kw in broad_keywords)
        explicit_no    = "referral is not required" in ans_lower
        referral_ok    = (explicit_yes or broad_match) and not explicit_no
        results["referral_recommended"] = "PASS" if referral_ok else "FAIL"
 
    # ── Check 4: At least one allowed dressing positively recommended ──────────
    allowed = test_case.get("allowed_dressings", [])
    if allowed:
        any_allowed = any(
            _is_dressing_recommended(form, generated_answer)
            for token in allowed
            for form in _dressing_surface_forms(token)
        )
        results["dressing_in_allowed_list"] = "PASS" if any_allowed else "FAIL"
 
    overall = "PASS" if all(v == "PASS" for v in results.values()) else "FAIL"
    results["overall"] = overall
    return results


# ──────────────────────────────────────────────────────────────────────────────
# Cell 6: Metric column definitions
# ──────────────────────────────────────────────────────────────────────────────
METRIC_COLS = [
    "llm_context_precision_with_reference",
    "context_recall",
    "faithfulness",
    "answer_relevancy",
]
METRIC_LABELS = [
    "Context precision",
    "Context recall",
    "Faithfulness",
    "Answer relevancy",
]
METRIC_COLORS = ["#185FA5", "#1D9E75", "#BA7517", "#D85A30"]


# ──────────────────────────────────────────────────────────────────────────────
# Cell 7: Main evaluation function (v3 — uses narrative_query for RAGAS)
# ──────────────────────────────────────────────────────────────────────────────
def run_evaluation(experiment_name: str, results_json: str):
    """
    Run RAGAS + rule-based safety evaluation for one experiment.
 
    v3 changes vs v2:
    ─────────────────
    1. narrative_query from API response is stored in each record.
    2. RAGAS SingleTurnSample uses narrative_query as user_input when
       available, falling back to the testset's user_input string.
       This fixes low AnswerRelevancy by giving RAGAS a semantically richer
       question that matches the style of the long clinical answer.
    3. safety checker is v3 (fixes false positives).
    """
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {experiment_name}")
    print(f"{'='*60}")
 
    safety_report_path = results_json.replace(".json", "_safety.csv")
 
    if os.path.isfile(results_json):
        with open(results_json, encoding="utf-8") as f:
            records = json.load(f)
        done = {r["index"] for r in records}
        print(f"Resuming: {len(records)}/{len(testset)} done")
    else:
        records = []
        done    = set()
 
    for idx, case in enumerate(testset):
        if idx in done:
            print(f"  [{idx+1:>2}/{len(testset)}] skip (done)")
            continue
 
        name = case.get("synthesizer_name", case.get("case_id", f"case_{idx}"))
        print(f"  [{idx+1:>2}/{len(testset)}] {name}")
        t0      = time.time()
        resp    = call_rag(case)
        elapsed = time.time() - t0
 
        answer  = resp.get("result", "")
        chunks  = resp.get("chunk_texts", [])
        retrieved_contexts = chunks if chunks else [answer]
 
        # v3: capture narrative_query from API response (new field)
        # Falls back to formatted user_input if server is pre-v3
        narrative_query = resp.get("narrative_query", "") or case.get("user_input", "")
 
        # ── Rule-based safety check ────────────────────────────────────────────
        safety = check_safety(answer, case)
        if safety:
            status = safety.get("overall", "N/A")
            print(f"       Safety: {status}")
 
        records.append({
            "index":               idx,
            "case_id":             case.get("case_id", f"case_{idx}"),
            "category":            case.get("category", "?"),
            "synthesizer_name":    name,
            "wound_type_expected": case.get("wound_type_expected", "?"),
            # [v3] store both user_input variants
            "user_input":          case.get("user_input", ""),          # structured (production format)
            "narrative_query":     narrative_query,                      # natural language (for RAGAS)
            "reference":           case.get("reference", ""),
            "reference_contexts":  case.get("reference_contexts", []),
            "retrieved_contexts":  retrieved_contexts,
            "answer":              answer,
            "confidence_label":    resp.get("confidence_label", "?"),
            "elapsed_sec":         round(elapsed, 1),
            "safety_checks":       safety,
        })
 
        with open(results_json, "w", encoding="utf-8") as f:
            json.dump(records, f, indent=2, ensure_ascii=False)
 
        time.sleep(1.0)
 
    # ── Save safety report ─────────────────────────────────────────────────────
    if has_v2:
        safety_rows = []
        for r in records:
            row = {
                "case_id":             r.get("case_id", ""),
                "category":            r.get("category", "?"),
                "wound_type_expected": r.get("wound_type_expected", "?"),
                "overall":             r.get("safety_checks", {}).get("overall", "N/A"),
            }
            for k, v in r.get("safety_checks", {}).items():
                if k != "overall":
                    row[k] = v
            safety_rows.append(row)
 
        safety_df = pd.DataFrame(safety_rows)
        safety_df.to_csv(safety_report_path, index=False)
 
        total  = len(safety_df)
        passed = (safety_df["overall"] == "PASS").sum() if "overall" in safety_df.columns else 0
        print(f"\n Safety Report: {passed}/{total} cases PASS ({100*passed//total if total else 0}%)")
        print(f"   Saved → {safety_report_path}")
 
        if "overall" in safety_df.columns:
            failed = safety_df[safety_df["overall"] == "FAIL"]
            if not failed.empty:
                print(f"\n   FAILED cases ({len(failed)}):")
                for _, row in failed.iterrows():
                    checks = [
                        k for k, v in row.items()
                        if k not in ("case_id", "category", "wound_type_expected", "overall")
                        and v == "FAIL"
                    ]
                    print(f"      {row['case_id']} — {', '.join(checks)}")
 
    # ── Build RAGAS dataset ────────────────────────────────────────────────────
    samples = []
    for r in records:
        if not r["answer"] or r["answer"].startswith("ERROR"):
            continue
 
        ref_contexts = r.get("reference_contexts", [])
        if not ref_contexts:
            print(f"   No reference_contexts for {r.get('case_id', '?')} — RAGAS recall will be 0")
 
        # [v3 FIX] use narrative_query as RAGAS user_input when available.
        # The narrative question is semantically closer to the long clinical
        # answer, which lifts AnswerRelevancy out of the false-low range.
        ragas_user_input = r.get("narrative_query") or r.get("user_input", "")
 
        samples.append(SingleTurnSample(
            user_input         = ragas_user_input,
            reference          = r["reference"],
            reference_contexts = [str(c) for c in ref_contexts],
            retrieved_contexts = [str(c) for c in r["retrieved_contexts"]],
            response           = r["answer"],
        ))
 
    print(f"\nRunning RAGAS scoring on {len(samples)} samples...")
    dataset = EvaluationDataset(samples)
    results = evaluate(dataset, metrics=[
        LLMContextPrecisionWithReference(llm=judge_llm),
        LLMContextRecall(llm=judge_llm),
        Faithfulness(llm=judge_llm),
        AnswerRelevancy(llm=judge_llm, embeddings=judge_emb),
    ])
 
    scores_df = results.to_pandas()
    for col in METRIC_COLS:
        if col not in scores_df.columns:
            scores_df[col] = float("nan")
 
    agg = {col: scores_df[col].mean() for col in METRIC_COLS}
 
    print(f"\n{'─'*40}")
    print(f"  RAGAS SCORES — {experiment_name}")
    print(f"{'─'*40}")
    for label, col in zip(METRIC_LABELS, METRIC_COLS):
        print(f"  {label:<25} {agg[col]:.4f}  ({agg[col]*100:.1f}%)")
    print(f"{'─'*40}")
 
    if HAS_MPL:
        fig, ax = plt.subplots(figsize=(8, 4))
        vals   = [agg[c] for c in METRIC_COLS]
        bars   = ax.bar(METRIC_LABELS, vals, color=METRIC_COLORS, alpha=0.85, zorder=3)
        ax.set_ylim(0, 1.0)
        ax.set_ylabel("Score")
        ax.set_title(f"RAGAS Metrics — {experiment_name}")
        ax.grid(axis="y", alpha=0.3, zorder=0)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9)
        chart_path = results_json.replace(".json", "_chart.png")
        fig.tight_layout()
        fig.savefig(chart_path, dpi=150)
        plt.close(fig)
        print(f"   Chart saved → {chart_path}")
 
    agg_df = pd.DataFrame([
        {"metric": lbl, "score": agg[col]}
        for lbl, col in zip(METRIC_LABELS, METRIC_COLS)
    ])
    return agg_df, scores_df

c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\GIGA\AppData\Local\Temp\ipykernel_25108\1571082393.py:29: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\GIGA\AppData\Local\Temp\ipykernel_25108\1571082393.py:29: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\GIGA\AppData\Local\T

✅ Loaded 28 test cases from wound_testset_v2.json
   ✅ v2 testset format detected — full evaluation enabled
      Category A: 8 cases
      Category B: 10 cases
      Category C: 6 cases
      Category D: 4 cases


C:\Users\GIGA\AppData\Local\Temp\ipykernel_25108\1571082393.py:54: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [2]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 8: Run evaluations
# ──────────────────────────────────────────────────────────────────────────────
# Change experiment_name and results_json for each run.
# Start the matching wound_app_XX.py server before running.

agg_df_00, full_df_00 = run_evaluation(
    experiment_name = "wound_ragas_eval_00_v3",
    results_json    = "../ragas_eval_00_v3/wound_ragas_ablation_results_00_v3.json",
)
agg_df_00




EXPERIMENT: wound_ragas_eval_00_v3
  [ 1/28] cat_a_type1_dry
       Safety: PASS
  [ 2/28] cat_a_type2_wet
       Safety: PASS
  [ 3/28] cat_a_type3_dry_infected
       Safety: PASS
  [ 4/28] cat_a_type4_wet_infected
       Safety: PASS
  [ 5/28] cat_a_type5_dry_necrotic
       Safety: PASS
  [ 6/28] cat_a_type6_wet_necrotic
       Safety: FAIL
  [ 7/28] cat_a_type7_dry_infected_necrotic
       Safety: PASS
  [ 8/28] cat_a_type8_wet_infected_necrotic
       Safety: PASS
  [ 9/28] cat_b_iodine_thyroid
       Safety: PASS
  [10/28] cat_b_silver_clean_granulating
       Safety: FAIL
  [11/28] cat_b_skin_tear_fragile
       Safety: FAIL
  [12/28] cat_b_npwt_necrotic_eschar
       Safety: PASS
  [13/28] cat_b_alginate_dry_wound
       Safety: PASS
  [14/28] cat_b_honey_dry_necrotic
       Safety: PASS
  [15/28] cat_b_postop_clean
       Safety: PASS
  [16/28] cat_b_burns_hand
       Safety: FAIL
  [17/28] cat_b_referral_type6
       Safety: PASS
  [18/28] cat_b_diabetic_foot
       Safety:

Evaluating:   7%|▋         | 8/112 [00:36<05:45,  3.32s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 112/112 [07:35<00:00,  4.07s/it]



────────────────────────────────────────
  RAGAS SCORES — wound_ragas_eval_00_v3
────────────────────────────────────────
  Context precision         0.9191  (91.9%)
  Context recall            0.6466  (64.7%)
  Faithfulness              0.6880  (68.8%)
  Answer relevancy          0.7387  (73.9%)
────────────────────────────────────────
   Chart saved → ../ragas_eval_00_v3/wound_ragas_ablation_results_00_v3_chart.png


,metric,score
0,Context precision,0.919107
1,Context recall,0.646577
2,Faithfulness,0.688044
3,Answer relevancy,0.738666


In [4]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 9: Eval 01
# ──────────────────────────────────────────────────────────────────────────────
agg_df_01, full_df_01 = run_evaluation(
    experiment_name = "wound_ragas_eval_01_v3",
    results_json    = "../ragas_eval_01_v3/wound_ragas_ablation_results_01_v3.json",
)
agg_df_01

# ──────────────────────────────────────────────────────────────────────────────



EXPERIMENT: wound_ragas_eval_01_v3
  [ 1/28] cat_a_type1_dry
       Safety: PASS
  [ 2/28] cat_a_type2_wet
       Safety: PASS
  [ 3/28] cat_a_type3_dry_infected
       Safety: PASS
  [ 4/28] cat_a_type4_wet_infected
       Safety: PASS
  [ 5/28] cat_a_type5_dry_necrotic
       Safety: PASS
  [ 6/28] cat_a_type6_wet_necrotic
       Safety: FAIL
  [ 7/28] cat_a_type7_dry_infected_necrotic
       Safety: PASS
  [ 8/28] cat_a_type8_wet_infected_necrotic
       Safety: PASS
  [ 9/28] cat_b_iodine_thyroid
       Safety: PASS
  [10/28] cat_b_silver_clean_granulating
       Safety: PASS
  [11/28] cat_b_skin_tear_fragile
       Safety: FAIL
  [12/28] cat_b_npwt_necrotic_eschar
       Safety: PASS
  [13/28] cat_b_alginate_dry_wound
       Safety: FAIL
  [14/28] cat_b_honey_dry_necrotic
       Safety: PASS
  [15/28] cat_b_postop_clean
       Safety: PASS
  [16/28] cat_b_burns_hand
       Safety: FAIL
  [17/28] cat_b_referral_type6
       Safety: PASS
  [18/28] cat_b_diabetic_foot
       Safety:

Evaluating: 100%|██████████| 112/112 [07:54<00:00,  4.24s/it]



────────────────────────────────────────
  RAGAS SCORES — wound_ragas_eval_01_v3
────────────────────────────────────────
  Context precision         0.9310  (93.1%)
  Context recall            0.7277  (72.8%)
  Faithfulness              0.6755  (67.6%)
  Answer relevancy          0.7318  (73.2%)
────────────────────────────────────────
   Chart saved → ../ragas_eval_01_v3/wound_ragas_ablation_results_01_v3_chart.png


,metric,score
0,Context precision,0.931002
1,Context recall,0.727749
2,Faithfulness,0.675539
3,Answer relevancy,0.731792


In [7]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 10: Eval 02
# ──────────────────────────────────────────────────────────────────────────────
agg_df_02, full_df_02 = run_evaluation(
    experiment_name = "wound_ragas_eval_02_v3",
    results_json    = "../ragas_eval_02_v3/wound_ragas_ablation_results_02_v3.json",
)
agg_df_02




EXPERIMENT: wound_ragas_eval_02_v3
  [ 1/28] cat_a_type1_dry
       Safety: FAIL
  [ 2/28] cat_a_type2_wet
       Safety: PASS
  [ 3/28] cat_a_type3_dry_infected
       Safety: FAIL
  [ 4/28] cat_a_type4_wet_infected
       Safety: PASS
  [ 5/28] cat_a_type5_dry_necrotic
       Safety: PASS
  [ 6/28] cat_a_type6_wet_necrotic
       Safety: FAIL
  [ 7/28] cat_a_type7_dry_infected_necrotic
       Safety: PASS
  [ 8/28] cat_a_type8_wet_infected_necrotic
       Safety: PASS
  [ 9/28] cat_b_iodine_thyroid
       Safety: FAIL
  [10/28] cat_b_silver_clean_granulating
       Safety: FAIL
  [11/28] cat_b_skin_tear_fragile
       Safety: FAIL
  [12/28] cat_b_npwt_necrotic_eschar
       Safety: PASS
  [13/28] cat_b_alginate_dry_wound
       Safety: PASS
  [14/28] cat_b_honey_dry_necrotic
       Safety: PASS
  [15/28] cat_b_postop_clean
       Safety: FAIL
  [16/28] cat_b_burns_hand
       Safety: FAIL
  [17/28] cat_b_referral_type6
       Safety: FAIL
  [18/28] cat_b_diabetic_foot
       Safety:

Evaluating: 100%|██████████| 112/112 [07:52<00:00,  4.22s/it]



────────────────────────────────────────
  RAGAS SCORES — wound_ragas_eval_02_v3
────────────────────────────────────────
  Context precision         0.8231  (82.3%)
  Context recall            0.5484  (54.8%)
  Faithfulness              0.7123  (71.2%)
  Answer relevancy          0.7268  (72.7%)
────────────────────────────────────────
   Chart saved → ../ragas_eval_02_v3/wound_ragas_ablation_results_02_v3_chart.png


,metric,score
0,Context precision,0.823125
1,Context recall,0.548427
2,Faithfulness,0.712256
3,Answer relevancy,0.726766


In [8]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 11: Comparison table
# ──────────────────────────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    "Metric": METRIC_LABELS,
    "Eval_v3_00 (baseline)":   [agg_df_00[agg_df_00.metric == lbl].score.values[0] for lbl in METRIC_LABELS],
    "Eval_v3_01 (+BM25)":      [agg_df_01[agg_df_01.metric == lbl].score.values[0] for lbl in METRIC_LABELS],
    "Eval_v3_02 (+reranker)":  [agg_df_02[agg_df_02.metric == lbl].score.values[0] for lbl in METRIC_LABELS],
})
comparison = comparison.set_index("Metric")
print(comparison.to_string())

                   Eval_v3_00 (baseline)  Eval_v3_01 (+BM25)  Eval_v3_02 (+reranker)
Metric                                                                              
Context precision               0.919107            0.931002                0.823125
Context recall                  0.646577            0.727749                0.548427
Faithfulness                    0.688044            0.675539                0.712256
Answer relevancy                0.738666            0.731792                0.726766
